# Pixel-time graph ablations on Kaggle

Compare the same pixel encoder, graph-time filter, and pixel decoder with **no graph** and a **uniform graph** only. No graph removes neighbour messages, not the spatial pixel CNN. Uniform fixes spatial weights, but keeps the learned self/spatial/temporal mixture.

Attach your private cache dataset, enable a GPU and Internet, and add an enabled Kaggle Secret named `GITHUB_TOKEN` (read-only access to the private repository). Never paste the token into this notebook.

**The ablation branch must be committed and pushed to GitHub before the clone cell will work.** Run cells in order. Full training is opt-in below. The smoke test is not an accuracy experiment.

In [ ]:
from pathlib import Path

BRANCH = 'wishaal/pixel-time-ablations-kaggle'
REPO_DIR = Path('/kaggle/working/GraphMLWildfire-ablations')
DATA_DIR = Path('/kaggle/input/datasets/wkanhai/zip-graphmlwildfire-train-cache/training_cache')
OUTPUT_DIR = Path('/kaggle/working/pixel_time_ablations')

FOLDS_TO_RUN = [0]  # Start with one fold. Use list(range(12)) across sessions for all folds.
SEEDS = [0]
MODES = ['none', 'uniform']
EPOCHS = 30  # Keep this equal across all variants.
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 1
PRECISION = 'auto'  # Native BF16 when supported; FP16 + gradient scaling on Tesla T4.
RUN_TRAINING = False  # Set True only after the smoke test succeeds.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Clone the code
Authentication is provided only through the child process environment, not the URL, notebook output, or saved Git config. An existing checkout must be clean and on the requested branch; this cell never overwrites or automatically updates it. To use newer code, use a fresh directory or deliberately update your checkout.

In [ ]:
import base64
import os
import subprocess
import sys

def git_output(*args):
    return subprocess.check_output(['git', '-C', str(REPO_DIR), *args], text=True).strip()

def clone_code():
    if REPO_DIR.exists():
        if git_output('branch', '--show-current') != BRANCH:
            raise RuntimeError('Existing checkout is on a different branch; use a new REPO_DIR.')
        if git_output('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Existing checkout has tracked edits; inspect them before proceeding.')
        print('Using existing checkout; not fetching updates.')
        return
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env = os.environ.copy()
    env.update({
        'GIT_TERMINAL_PROMPT': '0',
        'GIT_CONFIG_COUNT': '2',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'Authorization: Basic {auth}',
        'GIT_CONFIG_KEY_1': 'credential.helper',
        'GIT_CONFIG_VALUE_1': '',
    })
    subprocess.run([
        'git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH,
        'https://github.com/berkentekin/GraphMLWildfire.git', str(REPO_DIR),
    ], env=env, check=True)

clone_code()
CODE_COMMIT = git_output('rev-parse', 'HEAD')
print('Branch:', BRANCH)
print('Commit:', CODE_COMMIT)


## 2. Use Kaggle's existing GPU PyTorch
Do not run `uv sync` or upgrade PyTorch here. Install only missing runtime dependencies, then install this repository without resolving its training-machine version pins. Import errors other than missing top-level packages are surfaced rather than hidden.

In [ ]:
import importlib
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle notebook settings first.')
packages = ['numpy', 'h5py', 'rasterio', 'affine', 'tqdm', 'wandb', 'pandas']
missing = []
for name in packages:
    try:
        importlib.import_module(name)
    except ModuleNotFoundError as exc:
        if exc.name != name:
            raise
        missing.append(name)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(REPO_DIR),
], check=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
os.environ['WANDB_MODE'] = 'disabled'  # Local results only; no W&B login or upload.
os.environ['PYTHONUNBUFFERED'] = '1'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
from graphml_wildfire.train import resolve_precision
print('Training precision:', resolve_precision(PRECISION, torch.device('cuda')))


In [ ]:
import json
from graphml_wildfire.benchmark import YEARS
from graphml_wildfire.data.mmap import TARGETS
from graphml_wildfire.data.pixels import PIXEL_CHANNELS

fires = sorted(DATA_DIR.glob('*/*/features.npy'))
if not fires:
    raise FileNotFoundError(f'No cache found under {DATA_DIR}; check the attached dataset path.')
required = [f'{name}.npy' for name in TARGETS] + [f'pix_{name}.npy' for name in PIXEL_CHANNELS] + ['pix_centre.npy']
missing = [str(fire.parent / name) for fire in fires for name in required if not (fire.parent / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing cache files: {missing[:10]}')
for year in YEARS:
    count = sum(fire.parent.parent.name == str(year) for fire in fires)
    if not count:
        raise ValueError(f'Missing year {year}')
    print(year, 'fires:', count)
print('Total fires:', len(fires))
environment = {
    'branch': BRANCH, 'commit': CODE_COMMIT, 'data_dir': str(DATA_DIR),
    'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
    'python': sys.version, 'precision': PRECISION,
}
environment_file = OUTPUT_DIR / 'environment.json'
if environment_file.exists() and json.loads(environment_file.read_text()) != environment:
    raise ValueError('This OUTPUT_DIR contains a different code/environment setup; use a new OUTPUT_DIR.')
environment_file.write_text(json.dumps(environment, indent=2))


## 3. Smoke-test no graph and uniform graph on the actual GPU/cache
Each mode uses the full default architecture, one training batch (forward, backward, optimizer step), then one whole-fire validation sample. This does not train a useful model or produce benchmark AP. Initial cache statistics can take time.

If the test fails, stop. For FP16 non-finite gradients, try `PRECISION = 'fp32'` and rerun; use that same setting for all variants. If memory is insufficient, reduce `BATCH_SIZE` for all variants. The smoke test uses at most two training samples; full training can still need more memory.

In [ ]:
SMOKE_FILE = OUTPUT_DIR / 'smoke.json'
SMOKE_PASSED = False
subprocess.run([
    sys.executable, '-m', 'graphml_wildfire.smoke',
    '--data-dir', str(DATA_DIR), '--precision', PRECISION,
    '--fold', str(FOLDS_TO_RUN[0]), '--batch-size', str(min(2, BATCH_SIZE)),
    '--output', str(SMOKE_FILE),
    '--graph-modes', *MODES,
], cwd=REPO_DIR, check=True)
SMOKE_PASSED = True
SMOKE_SETTINGS = (str(DATA_DIR), PRECISION, FOLDS_TO_RUN[0], tuple(MODES))
print('Smoke test passed. You can now opt into training in the settings cell.')


## 4. Train and evaluate the matched comparison
Set `RUN_TRAINING = True` in the settings cell and rerun that cell, then this one. Defaults run **two 30-epoch models on fold 0**: no graph and uniform graph, sequentially on one GPU. This can take hours; measure one fold before planning all 12. A single fold is preliminary evidence, not a full benchmark.

Both variants use the same folds, seeds, batch size, epochs, feature inputs, crop, and precision. Neither has an attention network, and both have the same parameter count.

Best validation weights/config are saved whenever validation AP improves. These are **not full optimizer/scheduler resume checkpoints**. The cell skips exact runs already recorded as completed; an interrupted run without a result record starts from scratch. Export outputs before ending a session. For more folds, reuse the same settings and restore previous outputs to the same output directory if you want skipping/aggregation across sessions.

In [ ]:
from dataclasses import asdict
from graphml_wildfire.train import TrainConfig

RESULTS_FILE = OUTPUT_DIR / 'benchmark_runs.jsonl'
if not RUN_TRAINING:
    print('Training is disabled. Set RUN_TRAINING = True after the smoke test passes.')
else:
    if not globals().get('SMOKE_PASSED', False):
        raise RuntimeError('Run the smoke-test cell successfully first.')
    if SMOKE_SETTINGS != (str(DATA_DIR), PRECISION, FOLDS_TO_RUN[0], tuple(MODES)):
        raise RuntimeError('Cache/precision/initial fold/modes changed; rerun the smoke test first.')
    completed = {json.loads(line)['run_name'] for line in RESULTS_FILE.read_text().splitlines()} if RESULTS_FILE.exists() else set()
    for fold in FOLDS_TO_RUN:
        for seed in SEEDS:
            for mode in MODES:
                config = TrainConfig(
                    experiment_name=f'pixel_time_{mode}', graph_mode=mode,
                    fold=fold, seed=seed, epochs=EPOCHS,
                    batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE,
                    precision=PRECISION, compile=0, data_dir=str(DATA_DIR),
                    checkpoint_dir=str(OUTPUT_DIR / 'checkpoints'), results_file=str(RESULTS_FILE),
                )
                if config.run_name in completed:
                    print('Already complete:', config.run_name)
                    continue
                command = [sys.executable, '-m', 'graphml_wildfire.train']
                for key, value in asdict(config).items():
                    command.extend(['--' + key.replace('_', '-'), str(value)])
                print('Starting:', config.run_name, flush=True)
                subprocess.run(command, cwd=REPO_DIR, check=True)
                completed.add(config.run_name)


## 5. Results and paired differences
The main leaderboard metric is **`test_ap_centre128`**. `test_ap` scores the full benchmark crop. F1/IoU below use the existing 0.5 threshold. AP differences are absolute (0.01 = one percentage point).

Only complete no-graph/uniform-graph fold/seed pairs enter the comparison. The AP difference is uniform minus no graph. Mixing different settings in one summary is rejected. The per-seed summary reports sample standard deviation across folds, unavailable for a single fold. Do not claim improvement until the measured results support it.

In [ ]:
import pandas as pd
from IPython.display import display

if not RESULTS_FILE.exists():
    print('No benchmark results yet. Smoke-test losses are not benchmark results.')
else:
    rows = pd.read_json(RESULTS_FILE, lines=True).drop_duplicates('run_name', keep='last')
    rows = rows[rows.graph_mode.isin(MODES) & rows.fold.isin(FOLDS_TO_RUN) & rows.seed.isin(SEEDS)].copy()
    allowed = {'graph_mode', 'experiment_name', 'fold', 'seed'}
    for field in TrainConfig.__dataclass_fields__:
        if field not in allowed and rows[field].nunique() > 1:
            raise ValueError(f'Mixed {field} settings in results; use a separate OUTPUT_DIR per experiment.')
    if rows.duplicated(['graph_mode', 'fold', 'seed']).any():
        raise ValueError('Multiple runs for one mode/fold/seed; separate experiments before comparing.')
    columns = ['graph_mode', 'fold', 'seed', 'best_epoch', 'test_ap_centre128', 'test_ap', 'test_f1', 'test_iou', 'parameters', 'minutes', 'resolved_precision']
    table = rows[columns].sort_values(['fold', 'seed', 'graph_mode'])
    display(table)
    table.to_csv(OUTPUT_DIR / 'per_fold_results.csv', index=False)
    summary = rows.groupby(['graph_mode', 'seed']).agg(
        folds=('fold', 'nunique'), mean_ap=('test_ap_centre128', 'mean'), sample_std_ap=('test_ap_centre128', 'std'),
    ).reset_index()
    display(summary)
    summary.to_csv(OUTPUT_DIR / 'summary.csv', index=False, na_rep='not available')
    paired = rows.pivot(index=['fold', 'seed'], columns='graph_mode', values='test_ap_centre128').reindex(columns=MODES).dropna()
    paired['uniform_minus_none'] = paired['uniform'] - paired['none']
    display(paired)
    paired.to_csv(OUTPUT_DIR / 'paired_differences.csv')
    print('Complete matched fold/seed comparisons:', len(paired))


## 6. Preserve your results
Run this after the runs you want to keep. Download the archive or save a notebook version with outputs before the session ends. Keep the notebook private while it contains private-project code/output. The ZIP contains results, provenance, smoke diagnostics, and best weights/configs, not the training cache or GitHub secret.

In [ ]:
import shutil
from IPython.display import FileLink

archive = shutil.make_archive('/kaggle/working/pixel_time_ablations_results', 'zip', OUTPUT_DIR)
display(FileLink(archive))
